# Diamond Creek ETF Arbitrage Fund — Strategy Backtest v5

**Universe:** 40 leveraged ETFs from DC_Universe_Map_Filtered.xlsx (one per underlying, largest AUM)
**Capital:** $10M starting NAV
**Weighting:** Equal-weight across all active pairs, redistributed as pairs enter
**Rebalancing:** Conditional — shorts: trigger at 2% gross drift, hedge to 1%; longs: trigger at 1%, hedge to flat
**Costs:** IBKR commissions + 20bps slippage + IBKR borrow rates + tiered margin debit/credit interest
**Fees:** None (gross returns only)
**Runs:** 3×, 4×, 5×, 6× gross leverage


## Cell 1 — Setup


In [ ]:
# !pip install requests pandas numpy yfinance -q

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import requests
import time
import ftplib
import io
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

TRADING_DAYS = 252
norm_sym = lambda x: str(x).strip().upper().replace(".", "-")
print("Imports OK")


## Cell 2 — Configuration


In [ ]:
CFG = {
    "capital_usd":          10_000_000,
    "start_date":           "2023-01-01",
    "slippage_bps":         20,
    "ibkr_comm_per_share":  0.005,
    "ibkr_comm_min":        1.00,
    "ibkr_comm_max_pct":    0.005,
    "fallback_borrow_rate": 0.005,
    "beta_window":          252,
    "min_beta_days":        60,
    "min_trade_usd":        200,
    # Conditional rebalancing
    "short_rebal_trigger":  0.02,
    "short_rebal_target":   0.01,
    "long_rebal_trigger":   0.01,
    "long_rebal_target":    0.00,
    # IBKR margin financing
    "margin_debit_tiers": [
        (100_000,      0.0514),    # 0–100K
        (1_000_000,    0.0464),    # 100K–1M
        (50_000_000,   0.0439),    # 1M–50M
        (float("inf"), 0.0414),    # 50M+
    ],
    "credit_threshold":     10_000,
    "credit_rate":          0.0314,
    # FTP
    "skip_ftp":             False,
}

LEVERAGE_RUNS = [3, 4, 5, 6]

print(f"Capital: ${CFG['capital_usd']:,}  |  Start: {CFG['start_date']}")
print(f"Leverage runs: {LEVERAGE_RUNS}")
print(f"Rebal: shorts >{CFG['short_rebal_trigger']:.0%}→{CFG['short_rebal_target']:.0%}  |  "
      f"longs >{CFG['long_rebal_trigger']:.0%}→{CFG['long_rebal_target']:.0%}")
print(f"No beta filter — default nominal beta = 2.0 when rolling unavailable")





## Cell 3 — Universe (40 pairs from DC_Universe_Map_Filtered.xlsx)

Each pair = (leveraged_ETF, underlying). Ranked by AUM, one ETF per underlying.


In [ ]:
# 40 pairs from DC_Universe_Map_Filtered.xlsx with betas from etf_screened_today.csv
# (ETF, Underlying, Beta from OLS regression)
UNIVERSE_40 = [
    ("SOXL", "SOXX",  2.960),   #  1  $11,939M
    ("TSLL", "TSLA",  1.993),   #  2   $5,010M
    ("GDXU", "GDX",   3.043),   #  3   $4,089M
    ("NVDL", "NVDA",  1.988),   #  4   $3,934M
    ("KORU", "EWY",   2.931),   #  5   $1,155M
    ("GGLL", "GOOGL", 1.992),   #  6   $1,097M
    ("MUU",  "MU",    1.994),   #  7   $1,084M
    ("JNUG", "GDXJ",  1.974),   #  8     $910M
    ("BITX", "IBIT",  2.005),   #  9     $905M
    ("YINN", "FXI",   2.974),   # 10     $775M
    ("ETHU", "ETHA",  2.001),   # 11     $707M
    ("MSFU", "MSFT",  1.991),   # 12     $628M
    ("AMDL", "AMD",   1.997),   # 13     $566M
    ("LABU", "XBI",   2.988),   # 14     $557M
    ("PLTU", "PLTR",  1.992),   # 15     $490M
    ("CONL", "COIN",  1.987),   # 16     $470M
    ("METU", "META",  1.999),   # 17     $393M
    ("AMZU", "AMZN",  1.988),   # 18     $385M
    ("ERX",  "XLE",   1.988),   # 19     $315M
    ("GUSH", "XOP",   2.011),   # 20     $296M
    ("CWEB", "KWEB",  1.987),   # 21     $220M
    ("BMNU", "BMNR",  1.944),   # 22     $220M
    ("LITX", "LITE",  1.980),   # 23     $211M
    ("ASTX", "ASTS",  1.990),   # 24     $210M
    ("AVL",  "AVGO",  1.987),   # 25     $174M
    ("APPX", "APP",   1.984),   # 26     $170M
    ("CRCA", "CRCL",  1.978),   # 27     $167M
    ("UNHG", "UNH",   1.996),   # 28     $158M
    ("AAPU", "AAPL",  1.984),   # 29     $155M
    ("SOLT", "SOEZ",  2.014),   # 30     $139M
    ("CHAU", "ASHR",  1.967),   # 31     $131M
    ("INTW", "INTC",  1.988),   # 32     $121M
    ("BABX", "BABA",  2.006),   # 33     $117M
    ("XXRP", "XRPZ",  2.020),   # 34     $103M
    ("NEBX", "NBIS",  1.999),   # 35     $102M
    ("CRWG", "CRWV",  1.997),   # 36     $101M
    ("NFLU", "NFLX",  1.990),   # 37      $93M
    ("ROBN", "HOOD",  2.012),   # 38      $93M
    ("QBTX", "QBTS",  1.993),   # 39      $69M
    ("SMU",  "SMR",   1.986),   # 40      $68M
]

# Build lookup maps
ETF_BETA = {e: b for e, u, b in UNIVERSE_40}
ETF_TO_UND = {e: u for e, u, b in UNIVERSE_40}

ALL_TICKERS = set()
for e, u, b in UNIVERSE_40:
    ALL_TICKERS |= {e, u}
ALL_TICKERS.add("SPY")

print(f"Universe: {len(UNIVERSE_40)} pairs, {len(ALL_TICKERS)} unique tickers")
print(f"Beta range: {min(ETF_BETA.values()):.3f} – {max(ETF_BETA.values()):.3f}")
print(f"3x ETFs: {sum(1 for b in ETF_BETA.values() if b > 2.5)}")
print(f"2x ETFs: {sum(1 for b in ETF_BETA.values() if b <= 2.5)}")



## Cell 4 — Fetch IBKR Borrow Rates


In [ ]:
FTP_HOST = os.getenv("IBKR_FTP_HOST") or "ftp2.interactivebrokers.com"
FTP_USER = os.getenv("IBKR_FTP_USER") or "shortstock"
FTP_PASS = os.getenv("IBKR_FTP_PASS") or ""
BORROW_CACHE = Path("data/borrow_cache.csv")

all_etf_syms = [e for e, _, _ in UNIVERSE_40]

if CFG.get("skip_ftp"):
    print(f"[FTP] Skipped — flat {CFG['fallback_borrow_rate']:.2%}")
    BORROW_MAP = {e: CFG["fallback_borrow_rate"] for e in all_etf_syms}
else:
    BORROW_MAP = {}
    raw_df = None
    for attempt in range(1, 4):
        try:
            print(f"[FTP] Connecting (attempt {attempt}/3)...")
            ftp = ftplib.FTP(timeout=30)
            ftp.connect(FTP_HOST, 21)
            ftp.login(user=FTP_USER, passwd=FTP_PASS)
            ftp.set_pasv(True)
            buf = io.BytesIO()
            ftp.retrbinary("RETR usa.txt", buf.write)
            try: ftp.quit()
            except: pass
            text = buf.getvalue().decode("utf-8", errors="ignore")
            lines = [ln for ln in text.splitlines() if ln.strip()]
            hdr_idx = next(i for i, ln in enumerate(lines) if ln.startswith("#SYM|"))
            hdr = [c.strip().lstrip("#").lower() for c in lines[hdr_idx].split("|")]
            data = lines[hdr_idx + 1:]
            raw_df = pd.read_csv(io.StringIO("\n".join(data)), sep="|", header=None, engine="python")
            raw_df = raw_df.iloc[:, :min(len(hdr), raw_df.shape[1])]
            raw_df.columns = hdr[:raw_df.shape[1]]
            print(f"[FTP] Parsed {len(raw_df)} rows")
            BORROW_CACHE.parent.mkdir(parents=True, exist_ok=True)
            raw_df.to_csv(BORROW_CACHE, index=False)
            break
        except Exception as e:
            if attempt < 3: time.sleep(2**attempt)
            else:
                print(f"[FTP] Failed: {e}")
                if BORROW_CACHE.exists():
                    raw_df = pd.read_csv(BORROW_CACHE)
                    print(f"[FTP] Loaded cache ({len(raw_df)} rows)")

    if raw_df is not None:
        def _pr(x):
            try: return float(str(x).replace("%","").strip()) / 100
            except: return np.nan
        raw_df["sym"] = raw_df["sym"].astype(str).str.upper().str.strip()
        raw_df["net"] = (raw_df["feerate"].map(_pr) - raw_df["rebaterate"].map(_pr)).clip(lower=0)
        agg = raw_df.groupby("sym")["net"].max()
        found = missing = 0
        for e in all_etf_syms:
            if e in agg.index and pd.notna(agg[e]):
                BORROW_MAP[e] = float(agg[e])
                found += 1
            else:
                BORROW_MAP[e] = CFG["fallback_borrow_rate"]
                missing += 1
        print(f"[FTP] {found} from FTP, {missing} fallback")
    else:
        BORROW_MAP = {e: CFG["fallback_borrow_rate"] for e in all_etf_syms}
        print(f"[FTP] No data — flat {CFG['fallback_borrow_rate']:.2%}")

print(f"\nBorrow rates for universe:")
for e, _, _ in UNIVERSE_40[:10]:
    print(f"  {e:>6s}: {BORROW_MAP.get(e, 0):.2%}")
print(f"  ... ({len(BORROW_MAP)} total)")


## Cell 5 — Download Price Data


In [ ]:
def download_price(ticker, period="5y"):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    params = {"range": period, "interval": "1d", "events": "div,splits"}
    headers = {"User-Agent": "Mozilla/5.0 Chrome/120.0.0.0"}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=15)
        resp.raise_for_status()
        d = resp.json()["chart"]["result"][0]
        idx = pd.to_datetime(d["timestamp"], unit="s", utc=True).tz_convert("America/New_York").normalize()
        s = pd.Series(d["indicators"]["adjclose"][0]["adjclose"], index=idx, name=ticker, dtype=float).dropna()
        return s[~s.index.duplicated(keep="last")]
    except Exception:
        try:
            import yfinance as yf
            df = yf.download(ticker, period=period, progress=False, auto_adjust=True)
            if not df.empty:
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)
                s = df["Close"].squeeze()
                if isinstance(s, pd.DataFrame): s = s.iloc[:,0]
                s = pd.Series(s.values, index=s.index, name=ticker, dtype=float)
                return s.dropna()
        except: pass
        return pd.Series(dtype=float, name=ticker)

print(f"Downloading {len(ALL_TICKERS)} tickers...")
t0 = time.monotonic()
PRICES = {}
with ThreadPoolExecutor(max_workers=8) as pool:
    futs = {pool.submit(download_price, t, "5y"): t for t in sorted(ALL_TICKERS)}
    for i, f in enumerate(as_completed(futs)):
        t = futs[f]
        try:
            s = f.result()
            if not s.empty and len(s) > 30:
                PRICES[norm_sym(t)] = s
        except: pass
        if (i+1) % 20 == 0:
            print(f"  {i+1}/{len(ALL_TICKERS)} [{time.monotonic()-t0:.0f}s]")

print(f"Got {len(PRICES)}/{len(ALL_TICKERS)} [{time.monotonic()-t0:.1f}s]")

# Cleanup: force tz-naive pd.Series
for sym in list(PRICES.keys()):
    v = PRICES[sym]
    if isinstance(v, pd.DataFrame):
        if isinstance(v.columns, pd.MultiIndex):
            v.columns = v.columns.get_level_values(0)
        v = v.iloc[:, 0]
    if hasattr(v.index, 'tz') and v.index.tz is not None:
        v = v.tz_localize(None)
    PRICES[sym] = v

# Which pairs have full data?
AVAIL_PAIRS = [(e, u, b) for e, u, b in UNIVERSE_40 if e in PRICES and u in PRICES]
print(f"Pairs with price data: {len(AVAIL_PAIRS)}/{len(UNIVERSE_40)}")
missing = [(e, u) for e, u, b in UNIVERSE_40 if e not in PRICES or u not in PRICES]
if missing:
    print(f"Missing: {missing}")


## Cell 6 — Inception Dates


In [ ]:
# Inception = first date BOTH ETF and underlying have price data
# Beta is static from etf_screened_today.csv — no rolling computation needed

print("Computing inception dates...")
INCEPTION = {}

for etf, und, beta in UNIVERSE_40:
    if etf not in PRICES or und not in PRICES:
        print(f"  SKIP {etf}/{und}: missing price data")
        continue
    etf_dates = PRICES[etf].index
    und_dates = PRICES[und].index
    shared = etf_dates.intersection(und_dates).sort_values()
    if len(shared) < 2:
        print(f"  SKIP {etf}/{und}: < 2 shared dates")
        continue
    INCEPTION[etf] = shared[0]

print(f"\nPairs with data: {len(INCEPTION)}/{len(UNIVERSE_40)}")

print(f"\n{'#':>3s}  {'ETF':>6s}  {'Und':>8s}  {'Beta':>6s}  {'Inception':>12s}")
print("-" * 45)
for i, (etf, und, beta) in enumerate(UNIVERSE_40, 1):
    if etf in INCEPTION:
        print(f"{i:>3d}  {etf:>6s}  {und:>8s}  {beta:>6.3f}  {INCEPTION[etf].strftime('%Y-%m-%d')}")
    else:
        print(f"{i:>3d}  {etf:>6s}  {und:>8s}  {beta:>6.3f}  NO DATA")

start_ts = pd.Timestamp(CFG["start_date"])
n_at_start = sum(1 for etf in INCEPTION if INCEPTION[etf] <= start_ts)
print(f"\nActive at {CFG['start_date']}: {n_at_start}/{len(INCEPTION)} pairs")
if INCEPTION:
    print(f"Earliest: {min(INCEPTION.values()).strftime('%Y-%m-%d')}")
    print(f"Latest:   {max(INCEPTION.values()).strftime('%Y-%m-%d')}")



## Cell 7 — Cost & Margin Functions


In [ ]:
def trade_cost(tusd, px, cfg):
    """IBKR commission + slippage."""
    if abs(tusd) < 1 or px <= 0:
        return 0.0
    sh = abs(tusd) / px
    comm = max(cfg["ibkr_comm_min"],
               min(sh * cfg["ibkr_comm_per_share"], abs(tusd) * cfg["ibkr_comm_max_pct"]))
    return comm + abs(tusd) * cfg["slippage_bps"] / 10000

def margin_debit_interest(debit, tiers, days=1):
    """Daily interest on margin debit using IBKR tiered rates (360-day year)."""
    if debit <= 0:
        return 0.0
    remaining = debit
    interest = 0.0
    prev = 0.0
    for threshold, rate in tiers:
        amt = min(remaining, threshold - prev)
        if amt > 0:
            interest += amt * rate * days / 360
            remaining -= amt
        prev = threshold
        if remaining <= 0:
            break
    return interest

def margin_credit_interest(credit, threshold, rate, days=1):
    """Daily interest earned on credit balance above threshold."""
    if credit <= threshold:
        return 0.0
    return (credit - threshold) * rate * days / 360

# Sanity check
print("Margin debit check:")
for d in [100_000, 1_000_000, 10_000_000, 30_000_000]:
    daily = margin_debit_interest(d, CFG["margin_debit_tiers"])
    print(f"  ${d:>12,.0f} debit → ${daily:.2f}/day = ${daily*360:,.0f}/yr = {daily*360/d:.3%}")
print("\nMargin credit check:")
for c in [100_000, 5_000_000]:
    daily = margin_credit_interest(c, CFG["credit_threshold"], CFG["credit_rate"])
    print(f"  ${c:>12,.0f} credit → ${daily:.2f}/day = ${daily*360:,.0f}/yr")


## Cell 8 — Backtest Engine

**Cash accounting:** `NAV = cash + Σ(shares × price)`. Trade = `cash -= delta_shares × price`.

**Rebalancing — current implementation:**
- Compute targets: `pair_gross = NAV × leverage / n_active`, then beta-neutral split per pair
- First day, universe changes, or `gross_now <= 0` → full rebalance to all new targets
- Otherwise, per-position drift check every day
- For each short position: if `|current − target| / gross > 2%` → trade back to 1% of gross from target
- For each long position: if `|current − target| / gross > 1%` → trade back to target
- New pair entering (current=0, target≠0) opens automatically
- Pair leaving (target=0, current≠0) closes automatically

**Diagnostics added below:**
- Logs stale-price usage from SPY-calendar forward fills
- Logs full-rebalance reasons and turnover
- Logs actual vs target gross leverage drift
- Logs per-pair residual beta exposure and leg drift after trades
- Flags assumptions that can make the strategy look tighter or cleaner than reality


In [ ]:
def calc_pair_diagnostics(active_pairs, holdings_map, px_map, pair_gross_target):
    """Measure how tightly each pair is hedged after trading."""
    rows = []
    for etf, und, beta in active_pairs:
        p_und = px_map.get(und, 0.0)
        p_etf = px_map.get(etf, 0.0)
        if p_und <= 0 or p_etf <= 0:
            continue

        long_actual = max(holdings_map.get(und, 0.0), 0.0) * p_und
        short_actual = max(-holdings_map.get(etf, 0.0), 0.0) * p_etf
        actual_pair_gross = long_actual + short_actual

        if pair_gross_target > 0:
            hr = 1.0 / abs(beta)
            target_long = pair_gross_target / (1.0 + hr)
            target_short = pair_gross_target - target_long
            gross_drift_pct = abs(actual_pair_gross - pair_gross_target) / pair_gross_target
            leg_drift_pct = (abs(long_actual - target_long) + abs(short_actual - target_short)) / pair_gross_target
            beta_resid = long_actual - abs(beta) * short_actual
            beta_resid_pct = abs(beta_resid) / pair_gross_target
        else:
            target_long = target_short = 0.0
            gross_drift_pct = leg_drift_pct = beta_resid_pct = 0.0
            beta_resid = 0.0

        rows.append({
            "etf": etf,
            "und": und,
            "beta": beta,
            "target_long": target_long,
            "target_short": target_short,
            "actual_long": long_actual,
            "actual_short": short_actual,
            "actual_pair_gross": actual_pair_gross,
            "beta_resid": beta_resid,
            "beta_resid_pct": beta_resid_pct,
            "gross_drift_pct": gross_drift_pct,
            "leg_drift_pct": leg_drift_pct,
        })
    return rows


spy = PRICES.get("SPY")
start_ts = pd.Timestamp(CFG["start_date"])
tdays = spy.index[spy.index >= start_ts].sort_values()
print(f"Backtest: {tdays[0].strftime('%Y-%m-%d')} → {tdays[-1].strftime('%Y-%m-%d')} ({len(tdays)} days)")
print(f"Tradable pairs: {len(INCEPTION)}")
print("Diagnostics: stale prices, full-rebal causes, target-vs-actual gross, pair hedge residuals")

ALL_BT = {}
ALL_DIAG = {}

for gross_lev in LEVERAGE_RUNS:
    print(f"\n{'='*70}")
    print(f"  RUNNING {gross_lev}× GROSS")
    print(f"{'='*70}")

    cash = float(CFG["capital_usd"])
    holdings = {}   # sym → shares (negative = short)
    daily_rows = []
    tot_costs = tot_borrow = tot_margin_debit = tot_margin_credit = 0.0
    rebal_count = 0
    prev_active_set = set()

    diag = {
        "full_rebal_days": 0,
        "full_rebal_initial": 0,
        "full_rebal_universe": 0,
        "full_rebal_gross_reset": 0,
        "drift_rebal_days": 0,
        "days_with_stale_active": 0,
        "stale_active_leg_obs": 0,
        "stale_active_pair_obs": 0,
        "max_stale_days": 0,
        "turnover_usd": 0.0,
        "skipped_trade_notional": 0.0,
        "skipped_trade_count": 0,
        "planned_trade_count": 0,
        "executed_trade_count": 0,
        "target_gross_sum": 0.0,
        "actual_gross_sum": 0.0,
        "gross_gap_abs_sum": 0.0,
        "max_gross_gap_pct": 0.0,
        "pair_beta_resid_pct_sum": 0.0,
        "pair_leg_drift_pct_sum": 0.0,
        "pair_gross_drift_pct_sum": 0.0,
        "pair_diag_obs": 0,
        "max_pair_beta_resid_pct": 0.0,
        "max_pair_leg_drift_pct": 0.0,
        "max_pair_gross_drift_pct": 0.0,
        "max_trade_turnover": 0.0,
        "max_trade_turnover_date": None,
    }

    for di, today in enumerate(tdays):

        # ── Prices ──
        px = {}
        stale_days = {}
        stale_syms = set()
        for sym, s in PRICES.items():
            if today in s.index:
                px[sym] = float(s.loc[today])
                stale_days[sym] = 0
            else:
                prior = s[s.index <= today]
                if not prior.empty:
                    px[sym] = float(prior.iloc[-1])
                    lag = int((today - prior.index[-1]).days)
                    stale_days[sym] = lag
                    if lag > 0:
                        stale_syms.add(sym)

        # ── NAV ──
        mtm = sum(sh * px.get(sym, 0) for sym, sh in holdings.items())
        nav = cash + mtm
        long_n = sum(sh * px.get(sym, 0) for sym, sh in holdings.items() if sh > 0)
        short_n = sum(abs(sh) * px.get(sym, 0) for sym, sh in holdings.items() if sh < 0)
        gross_now = long_n + short_n

        # ── Daily borrow ──
        borrow_d = 0.0
        for sym, sh in holdings.items():
            if sh < 0:
                borrow_d += abs(sh) * px.get(sym, 0) * BORROW_MAP.get(sym, CFG["fallback_borrow_rate"]) / TRADING_DAYS
        cash -= borrow_d
        tot_borrow += borrow_d

        # ── Margin interest on cash balance ──
        margin_debit_d = margin_credit_d = 0.0
        if cash < 0:
            margin_debit_d = margin_debit_interest(abs(cash), CFG["margin_debit_tiers"])
            cash -= margin_debit_d
            tot_margin_debit += margin_debit_d
        elif cash > CFG["credit_threshold"]:
            margin_credit_d = margin_credit_interest(cash, CFG["credit_threshold"], CFG["credit_rate"])
            cash += margin_credit_d
            tot_margin_credit += margin_credit_d

        # Recompute NAV after costs
        mtm = sum(sh * px.get(sym, 0) for sym, sh in holdings.items())
        nav = cash + mtm

        # ── Active pairs ──
        active_pairs = []
        for etf, und, beta in UNIVERSE_40:
            if etf not in INCEPTION or INCEPTION[etf] > today:
                continue
            if etf not in px or und not in px:
                continue
            active_pairs.append((etf, und, beta))
        n_active = len(active_pairs)
        pair_gross = tgt_gross = 0.0

        stale_active_legs = sum(int(etf in stale_syms) + int(und in stale_syms) for etf, und, _ in active_pairs)
        stale_active_pairs = sum(int(etf in stale_syms or und in stale_syms) for etf, und, _ in active_pairs)
        max_stale_days_active = max([max(stale_days.get(etf, 0), stale_days.get(und, 0)) for etf, und, _ in active_pairs], default=0)
        if stale_active_pairs > 0:
            diag["days_with_stale_active"] += 1
        diag["stale_active_leg_obs"] += stale_active_legs
        diag["stale_active_pair_obs"] += stale_active_pairs
        diag["max_stale_days"] = max(diag["max_stale_days"], max_stale_days_active)

        # ── Target shares (equal-weight, beta-neutral) ──
        tgt_gross = nav * gross_lev
        tgt_shares = {}
        if n_active > 0 and tgt_gross > 0:
            pair_gross = tgt_gross / n_active
            for etf, und, bv in active_pairs:
                hr = 1.0 / abs(bv)
                long_usd = pair_gross / (1.0 + hr)
                short_usd = pair_gross - long_usd
                p_und = px.get(und, 0)
                p_etf = px.get(etf, 0)
                if p_und > 0:
                    tgt_shares[und] = tgt_shares.get(und, 0.0) + long_usd / p_und
                if p_etf > 0:
                    tgt_shares[etf] = tgt_shares.get(etf, 0.0) - short_usd / p_etf

        # ── Did the universe change? ──
        active_set = frozenset(e for e, u, b in active_pairs)
        universe_changed = (active_set != prev_active_set)
        prev_active_set = active_set

        # ── Determine which symbols to trade ──
        all_syms = set(tgt_shares.keys()) | set(holdings.keys())
        trades = {}   # sym → target_shares to trade to

        full_rebal = (di == 0 or universe_changed or gross_now <= 0)
        full_rebal_reason = None
        if full_rebal:
            diag["full_rebal_days"] += 1
            if di == 0:
                diag["full_rebal_initial"] += 1
                full_rebal_reason = "initial"
            elif universe_changed:
                diag["full_rebal_universe"] += 1
                full_rebal_reason = "universe"
            else:
                diag["full_rebal_gross_reset"] += 1
                full_rebal_reason = "gross<=0"

        for sym in all_syms:
            tgt_sh = tgt_shares.get(sym, 0.0)
            cur_sh = holdings.get(sym, 0.0)
            p = px.get(sym, 0)
            if p <= 0:
                continue

            # Full rebal: trade everything to target
            if full_rebal:
                trades[sym] = tgt_sh
                continue

            tgt_notional = tgt_sh * p
            cur_notional = cur_sh * p
            drift_usd = abs(cur_notional - tgt_notional)

            # Position should be closed
            if tgt_sh == 0.0 and cur_sh != 0.0:
                trades[sym] = 0.0
                continue

            drift_pct = drift_usd / gross_now if gross_now > 0 else np.inf

            if tgt_notional < 0:
                # SHORT: trigger at 2% of gross, hedge back to 1%
                if drift_pct > CFG["short_rebal_trigger"]:
                    allowed_drift = CFG["short_rebal_target"] * gross_now
                    sign = 1.0 if cur_notional > tgt_notional else -1.0
                    exec_notional = tgt_notional + sign * allowed_drift
                    trades[sym] = exec_notional / p
            elif tgt_notional > 0:
                # LONG: trigger at 1% of gross, hedge back to flat (target)
                if drift_pct > CFG["long_rebal_trigger"]:
                    trades[sym] = tgt_sh

        # ── Execute trades ──
        trade_turnover = 0.0
        skipped_trade_notional = 0.0
        trades_executed = 0
        if trades:
            rebal_count += 1
            if not full_rebal:
                diag["drift_rebal_days"] += 1
            dc = 0.0
            diag["planned_trade_count"] += len(trades)
            for sym, new_sh in trades.items():
                cur_sh = holdings.get(sym, 0.0)
                delta = new_sh - cur_sh
                p = px.get(sym, 0)
                trade_usd = abs(delta * p)
                if p <= 0 or trade_usd < CFG["min_trade_usd"]:
                    if trade_usd > 0:
                        skipped_trade_notional += trade_usd
                        diag["skipped_trade_notional"] += trade_usd
                        diag["skipped_trade_count"] += 1
                    continue

                cash -= delta * p
                c = trade_cost(trade_usd, p, CFG)
                cash -= c
                dc += c
                trade_turnover += trade_usd
                trades_executed += 1

                if abs(new_sh) < 0.001:
                    holdings.pop(sym, None)
                else:
                    holdings[sym] = new_sh
            tot_costs += dc
            diag["turnover_usd"] += trade_turnover
            diag["executed_trade_count"] += trades_executed
            if trade_turnover > diag["max_trade_turnover"]:
                diag["max_trade_turnover"] = trade_turnover
                diag["max_trade_turnover_date"] = today

        # ── Record ──
        mtm = sum(sh * px.get(sym, 0) for sym, sh in holdings.items())
        nv = cash + mtm
        l = sum(sh * px.get(sym, 0) for sym, sh in holdings.items() if sh > 0)
        s = sum(abs(sh) * px.get(sym, 0) for sym, sh in holdings.items() if sh < 0)
        actual_gross = l + s
        actual_lev = actual_gross / nv if nv > 0 else np.nan
        gross_gap_pct = (actual_gross - tgt_gross) / tgt_gross if tgt_gross > 0 else np.nan
        diag["target_gross_sum"] += max(tgt_gross, 0.0)
        diag["actual_gross_sum"] += max(actual_gross, 0.0)
        if pd.notna(gross_gap_pct):
            diag["gross_gap_abs_sum"] += abs(gross_gap_pct)
            diag["max_gross_gap_pct"] = max(diag["max_gross_gap_pct"], abs(gross_gap_pct))

        pair_diag = calc_pair_diagnostics(active_pairs, holdings, px, pair_gross)
        pair_beta_resid_abs = sum(abs(r["beta_resid"]) for r in pair_diag)
        pair_beta_resid_signed = sum(r["beta_resid"] for r in pair_diag)
        pair_beta_resid_pct_nav = pair_beta_resid_abs / nv if nv > 0 else np.nan
        portfolio_beta_resid_pct_nav = pair_beta_resid_signed / nv if nv > 0 else np.nan
        pair_leg_drift_avg = np.mean([r["leg_drift_pct"] for r in pair_diag]) if pair_diag else np.nan
        pair_leg_drift_max = np.max([r["leg_drift_pct"] for r in pair_diag]) if pair_diag else np.nan
        pair_beta_resid_max = np.max([r["beta_resid_pct"] for r in pair_diag]) if pair_diag else np.nan
        pair_gross_drift_avg = np.mean([r["gross_drift_pct"] for r in pair_diag]) if pair_diag else np.nan
        pair_gross_drift_max = np.max([r["gross_drift_pct"] for r in pair_diag]) if pair_diag else np.nan

        if pair_diag:
            diag["pair_beta_resid_pct_sum"] += float(np.nansum([r["beta_resid_pct"] for r in pair_diag]))
            diag["pair_leg_drift_pct_sum"] += float(np.nansum([r["leg_drift_pct"] for r in pair_diag]))
            diag["pair_gross_drift_pct_sum"] += float(np.nansum([r["gross_drift_pct"] for r in pair_diag]))
            diag["pair_diag_obs"] += len(pair_diag)
            diag["max_pair_beta_resid_pct"] = max(diag["max_pair_beta_resid_pct"], float(np.nanmax([r["beta_resid_pct"] for r in pair_diag])))
            diag["max_pair_leg_drift_pct"] = max(diag["max_pair_leg_drift_pct"], float(np.nanmax([r["leg_drift_pct"] for r in pair_diag])))
            diag["max_pair_gross_drift_pct"] = max(diag["max_pair_gross_drift_pct"], float(np.nanmax([r["gross_drift_pct"] for r in pair_diag])))

        daily_rows.append({
            "date": today,
            "nav": nv,
            "cash": cash,
            "long_notional": l,
            "short_notional": s,
            "gross_notional": actual_gross,
            "net_notional": l - s,
            "target_gross_notional": tgt_gross,
            "gross_gap_pct": gross_gap_pct,
            "actual_leverage": actual_lev,
            "n_positions": len(holdings),
            "n_longs": sum(1 for sh in holdings.values() if sh > 0),
            "n_shorts": sum(1 for sh in holdings.values() if sh < 0),
            "n_active_pairs": n_active,
            "full_rebal": int(full_rebal),
            "universe_changed": int(universe_changed),
            "stale_symbols": len(stale_syms),
            "stale_active_legs": stale_active_legs,
            "stale_active_pairs": stale_active_pairs,
            "max_stale_days_active": max_stale_days_active,
            "trade_turnover": trade_turnover,
            "trades_planned": len(trades),
            "trades_executed": trades_executed,
            "skipped_trade_notional": skipped_trade_notional,
            "pair_beta_resid_pct_nav": pair_beta_resid_pct_nav,
            "portfolio_beta_resid_pct_nav": portfolio_beta_resid_pct_nav,
            "pair_leg_drift_pct_avg": pair_leg_drift_avg,
            "pair_leg_drift_pct_max": pair_leg_drift_max,
            "pair_beta_resid_pct_max": pair_beta_resid_max,
            "pair_gross_drift_pct_avg": pair_gross_drift_avg,
            "pair_gross_drift_pct_max": pair_gross_drift_max,
            "cum_costs": tot_costs,
            "cum_borrow": tot_borrow,
            "cum_margin_debit": tot_margin_debit,
            "cum_margin_credit": tot_margin_credit,
            "daily_borrow": borrow_d,
            "cash_balance": cash,
        })

        if di % 63 == 0:
            ds = today.strftime("%Y-%m-%d")
            reason_txt = f" | Full={full_rebal_reason}" if full_rebal else ""
            print(f"  {ds}  NAV=${nv:>12,.0f}  Gross=${actual_gross:>12,.0f}  "
                  f"L=${l:>10,.0f}  S=${s:>10,.0f}  Cash=${cash:>10,.0f}  Pairs={n_active:>2d}")
            print(f"             TargetGross=${tgt_gross:>12,.0f}  GrossGap={gross_gap_pct:+.2%}  "
                  f"Turnover=${trade_turnover:>10,.0f}  StalePairs={stale_active_pairs:>2d}"
                  f"  PairLegDriftMax={pair_leg_drift_max if pd.notna(pair_leg_drift_max) else 0.0:.2%}"
                  f"  PairBetaResidMax={pair_beta_resid_max if pd.notna(pair_beta_resid_max) else 0.0:.2%}{reason_txt}")

    bt = pd.DataFrame(daily_rows).set_index("date")
    bt.index = pd.to_datetime(bt.index)
    ALL_BT[gross_lev] = bt
    ALL_DIAG[gross_lev] = diag

    n_days = len(bt)
    pair_obs = max(diag["pair_diag_obs"], 1)
    worst_turnover_date = diag["max_trade_turnover_date"].strftime("%Y-%m-%d") if diag["max_trade_turnover_date"] is not None else "n/a"

    print(f"\n  {gross_lev}× done | Rebals: {rebal_count} | Costs: ${tot_costs:,.0f} | Borrow: ${tot_borrow:,.0f}")
    print(f"  Margin debit: ${tot_margin_debit:,.0f} | Credit: ${tot_margin_credit:,.0f}")
    print(f"  Avg pairs: {bt['n_active_pairs'].mean():.1f}")
    print(f"  Avg L/S: ${bt['long_notional'].mean():,.0f} / ${bt['short_notional'].mean():,.0f}")
    print(f"  Avg cash: ${bt['cash'].mean():,.0f}")
    print("  Diagnostics summary:")
    print(f"    Full rebalance days: {diag['full_rebal_days']} ({diag['full_rebal_days']/n_days:.1%}) | "
          f"initial={diag['full_rebal_initial']}, universe={diag['full_rebal_universe']}, gross<=0={diag['full_rebal_gross_reset']}")
    print(f"    Drift rebalance days: {diag['drift_rebal_days']} ({diag['drift_rebal_days']/n_days:.1%})")
    print(f"    Active pairs with stale prices: {diag['days_with_stale_active']} days ({diag['days_with_stale_active']/n_days:.1%}) | "
          f"avg stale active legs/day={diag['stale_active_leg_obs']/n_days:.2f} | max stale age={diag['max_stale_days']}d")
    print(f"    Avg target gross=${diag['target_gross_sum']/n_days:,.0f} | Avg actual gross=${diag['actual_gross_sum']/n_days:,.0f} | "
          f"Avg abs gross gap={diag['gross_gap_abs_sum']/n_days:.2%} | Max abs gross gap={diag['max_gross_gap_pct']:.2%}")
    print(f"    Avg turnover/day=${diag['turnover_usd']/n_days:,.0f} | Max turnover=${diag['max_trade_turnover']:,.0f} on {worst_turnover_date}")
    print(f"    Skipped tiny trades: {diag['skipped_trade_count']} | skipped notional=${diag['skipped_trade_notional']:,.0f}")
    print(f"    Mean pair leg drift={diag['pair_leg_drift_pct_sum']/pair_obs:.2%} | Max pair leg drift={diag['max_pair_leg_drift_pct']:.2%}")
    print(f"    Mean pair gross drift={diag['pair_gross_drift_pct_sum']/pair_obs:.2%} | Max pair gross drift={diag['max_pair_gross_drift_pct']:.2%}")
    print(f"    Mean pair beta residual={diag['pair_beta_resid_pct_sum']/pair_obs:.2%} | Max pair beta residual={diag['max_pair_beta_resid_pct']:.2%}")
    print("  Potential reality gaps to inspect:")
    print("    - Trades are marked and executed using the same daily close; this can overstate achievable hedge quality.")
    print("    - SPY drives the calendar, so missing ETF/underlying closes are forward-filled and can suppress apparent drift.")
    print("    - Universe additions trigger full-book rebalances, which can keep pairs tighter than a looser production process.")
    print("    - Hedge sizing uses static betas; real ETF beta and tracking can move materially through time.")
    print("    - End-of-day only simulation ignores intraday gap risk, borrow recalls, locates, and execution capacity.")

print(f"\n{'='*70}")
print(f"  ALL RUNS COMPLETE")
print(f"{'='*70}")



## Cell 9 — Performance Summary


In [ ]:
def perf(nav):
    r = nav.pct_change().dropna()
    ny = len(nav) / TRADING_DAYS
    tot = nav.iloc[-1] / nav.iloc[0] - 1
    cagr = (1 + tot) ** (1 / ny) - 1
    vol = r.std() * np.sqrt(TRADING_DAYS)
    sh = cagr / vol if vol > 0 else 0
    dd = (nav - nav.cummax()) / nav.cummax(); mdd = dd.min()
    cal = cagr / abs(mdd) if mdd != 0 else 0
    neg = r[r < 0]; dv = neg.std() * np.sqrt(TRADING_DAYS) if len(neg) > 0 else 0
    sor = cagr / dv if dv > 0 else 0
    mo = nav.resample("ME").last().pct_change().dropna(); wr = (mo > 0).mean()
    return {"Total Return": f"{tot:.2%}", "CAGR": f"{cagr:.2%}", "Vol": f"{vol:.2%}",
            "Sharpe": f"{sh:.2f}", "Sortino": f"{sor:.2f}", "Max DD": f"{mdd:.2%}",
            "Calmar": f"{cal:.2f}", "Monthly Win%": f"{wr:.1%}",
            "Final NAV": f"${nav.iloc[-1]:,.0f}", "P&L": f"${nav.iloc[-1]-nav.iloc[0]:,.0f}"}

for lev in LEVERAGE_RUNS:
    bt = ALL_BT[lev]
    print(f"\n{'='*60}")
    print(f"  {lev}× GROSS (no fees)")
    print(f"{'='*60}")
    for k, v in perf(bt["nav"]).items():
        print(f"  {k:20s}: {v}")
    print(f"\n  Txn Costs:     ${bt['cum_costs'].iloc[-1]:,.0f}")
    print(f"  Borrow Costs:  ${bt['cum_borrow'].iloc[-1]:,.0f}")
    print(f"  Margin Debit:  ${bt['cum_margin_debit'].iloc[-1]:,.0f}")
    print(f"  Margin Credit: ${bt['cum_margin_credit'].iloc[-1]:,.0f}")
    print(f"  Avg Lev:       {bt['gross_notional'].mean()/bt['nav'].mean():.2f}x")
    print(f"  Avg Cash:      ${bt['cash'].mean():,.0f}")
    print(f"  Avg Pairs:     {bt['n_active_pairs'].mean():.1f}")
    print(f"  Avg Pos:       {bt['n_longs'].mean():.0f}L / {bt['n_shorts'].mean():.0f}S")



## Cell 10 — Visualizations


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

colors = {3: "#2196F3", 4: "#4CAF50", 5: "#FF9800", 6: "#F44336"}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Diamond Creek v5 — Multi-Leverage ($10M, EW, No Fees)", fontsize=13, fontweight="bold")

ax = axes[0, 0]
for lev in LEVERAGE_RUNS:
    ax.plot(ALL_BT[lev].index, ALL_BT[lev]["nav"]/1e6, label=f"{lev}×", color=colors[lev], lw=1.5)
ax.axhline(CFG["capital_usd"]/1e6, color="gray", ls="--", alpha=0.5)
ax.set_ylabel("NAV ($M)"); ax.set_title("NAV"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for lev in LEVERAGE_RUNS:
    dd = (ALL_BT[lev]["nav"]-ALL_BT[lev]["nav"].cummax())/ALL_BT[lev]["nav"].cummax()
    ax.fill_between(dd.index, dd*100, 0, alpha=0.25, color=colors[lev], label=f"{lev}×")
ax.set_ylabel("DD (%)"); ax.set_title("Drawdowns"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
bt0 = ALL_BT[LEVERAGE_RUNS[0]]
ax.plot(bt0.index, bt0["n_active_pairs"], color="#9C27B0", lw=1.5, label="Active pairs")
ax.plot(bt0.index, bt0["n_longs"], color="#2196F3", lw=0.8, alpha=0.7, label="# Longs")
ax.plot(bt0.index, bt0["n_shorts"], color="#F44336", lw=0.8, alpha=0.7, label="# Shorts")
ax.set_ylabel("Count"); ax.set_title("Universe Ramp"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
for lev in LEVERAGE_RUNS:
    ax.plot(ALL_BT[lev].index, ALL_BT[lev]["cash"]/1e6,
            label=f"{lev}×", color=colors[lev], lw=1, alpha=0.7)
ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("Cash ($M)"); ax.set_title("Cash Balance"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

for ax in axes.flat:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
fig.suptitle("Leverage & Costs", fontsize=13, fontweight="bold")
ax = axes[0]
for lev in LEVERAGE_RUNS:
    bt = ALL_BT[lev]
    ax.plot(bt.index, bt["gross_notional"]/bt["nav"], label=f"{lev}×",
            color=colors[lev], lw=0.8, alpha=0.7)
ax.set_ylabel("Actual Lev (x)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax = axes[1]
for lev in LEVERAGE_RUNS:
    bt = ALL_BT[lev]
    tc = bt["cum_costs"]+bt["cum_borrow"]+bt["cum_margin_debit"]-bt["cum_margin_credit"]
    ax.plot(bt.index, tc/1e3, label=f"{lev}×", color=colors[lev], lw=1)
ax.set_ylabel("Cum Costs ($K)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.tight_layout(); plt.show()



## Cell 11 — Monthly Return Tables


In [ ]:
def monthly_table(nav):
    mo = nav.resample("ME").last().pct_change().dropna().to_frame("r")
    mo["year"], mo["month"] = mo.index.year, mo.index.month
    piv = mo.pivot(index="year", columns="month", values="r")
    piv.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][:len(piv.columns)]
    yr = nav.resample("YE").last().pct_change().dropna(); yr.index = yr.index.year
    piv["Annual"] = yr
    return piv

for lev in LEVERAGE_RUNS:
    print(f"\n{'='*80}")
    print(f"  {lev}× Monthly Returns")
    print(f"{'='*80}")
    print(monthly_table(ALL_BT[lev]["nav"]).map(
        lambda x: f"{x:+.2%}" if pd.notna(x) else "").to_string())


## Cell 12 — Leverage Comparison


In [ ]:
rows = []
for lev in LEVERAGE_RUNS:
    bt = ALL_BT[lev]; ng = bt["nav"]; ny = len(ng)/TRADING_DAYS
    tot = ng.iloc[-1]/ng.iloc[0]-1; cagr = (1+tot)**(1/ny)-1
    vol = ng.pct_change().dropna().std()*np.sqrt(TRADING_DAYS)
    sh = cagr/vol if vol > 0 else 0
    dd = ((ng-ng.cummax())/ng.cummax()).min()
    rows.append({
        "Lev": f"{lev}×", "CAGR": f"{cagr:.2%}", "Sharpe": f"{sh:.2f}", "Vol": f"{vol:.2%}",
        "Max DD": f"{dd:.2%}", "Final NAV": f"${ng.iloc[-1]:,.0f}",
        "Txn": f"${bt['cum_costs'].iloc[-1]:,.0f}",
        "Borrow": f"${bt['cum_borrow'].iloc[-1]:,.0f}",
        "Margin": f"${bt['cum_margin_debit'].iloc[-1]-bt['cum_margin_credit'].iloc[-1]:,.0f}",
        "Avg Pairs": f"{bt['n_active_pairs'].mean():.1f}",
    })
print("\n" + "="*110 + "\n  LEVERAGE COMPARISON (no fees)\n" + "="*110)
print(pd.DataFrame(rows).set_index("Lev").to_string())


## Cell 13 — Export


In [ ]:
out = Path("data/backtest"); out.mkdir(parents=True, exist_ok=True)
for lev in LEVERAGE_RUNS:
    ALL_BT[lev].to_csv(out / f"backtest_nav_{lev}x.csv")
pd.DataFrame(rows).set_index("Lev").to_csv(out / "leverage_comparison.csv")
print(f"Saved to {out}/")
